In [1]:
import numpy as np
import pymc as pm
import arviz as az
from scipy.stats import norm

from pybridgesampling import bridge_sample

In [2]:
# Observed Data
k = 2
n = 10
# Prior parameters for Beta
alpha = beta = 1

with pm.Model() as model:
    p = pm.Beta('p', alpha = 1., beta = 1.)
    obs = pm.Binomial('obs', p = p, n = n, observed = k)
    trace = pm.sample(idata_kwargs = dict(include_transformed = True), random_seed = 0)

# Exact Calculation
m_l = 1/(n + 1)
print('The analytical solution of the marginal likelihood is %.5f'%(m_l))
logml_dict = bridge_sample(model, trace, random_seed = 1)
print('The Bridge Sampling estimatation is %.5f'%(np.exp(logml_dict['logml'])))

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [p]


Output()

Sampling 3 chains for 1_000 tune and 1_000 draw iterations (3_000 + 3_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


The analytical solution of the marginal likelihood is 0.09091
The Bridge Sampling estimatation is 0.09094


In [3]:
mu, tau2, sigma2 = 0, 0.5, 1

n = 20
theta = norm.rvs(mu, np.sqrt(tau2), size=n)
y = norm.rvs(theta, np.sqrt(sigma2), size=n)

# set prior parameters
mu0, tau20 = 0, 1
alpha = beta = 1.

# Same sample from R, so it is eaiser to compare
y = np.array([1.19365332,  1.95745331, -0.72161754, -1.87380833, -1.16928239,  0.51960853,
             -0.03610041,  0.42508815,  0.41119221, -0.81236980,  0.72967357,  3.48186722,
              2.31126381,  2.00029422, -0.27643507,  1.06882370, -0.95083599, -1.89651101,
              2.56019737,  0.23703060])

with pm.Model() as H0:
    tau0 = pm.InverseGamma('tau', alpha, beta)
    theta0 = pm.Normal('theta', 0, np.sqrt(tau0), shape=n)
    obs0 = pm.Normal('y', theta0, np.sqrt(sigma2), observed=y)
    trace0 = pm.sample(10000, idata_kwargs = dict(include_transformed = True), random_seed = 0)

with pm.Model() as H1:
    mu1 = pm.Normal('mu', mu0, np.sqrt(tau20))
    tau1 = pm.InverseGamma('tau', alpha, beta)
    theta1 = pm.Normal('theta', mu1, np.sqrt(tau1), shape=n)
    obs1 = pm.Normal('y', theta1, np.sqrt(sigma2), observed=y)
    trace1 = pm.sample(10000, idata_kwargs = dict(include_transformed = True), random_seed = 0)

logml0 = bridge_sample(H0, trace0, random_seed = 1)
print('Bridge sampling estimate of the log marginal likelihood of H0: %.5f'
      %(logml0['logml']))
print('The percentage errors of the estimation is: %.5f'
      %(logml0['coefficient_of_variation']*100))

logml1 = bridge_sample(H1, trace1, random_seed = 1)
print('Bridge sampling estimate of the log marginal likelihood of H1: %.5f'
      %(logml1['logml']))
print('The percentage errors of the estimation is: %.5f'
      %(logml1['coefficient_of_variation']*100))

BF01 = np.exp(logml0['logml'] - logml1['logml'])
print('The estimated Bayes factor in favor of H0 over H1 is equal to: %.5f'%(BF01))

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [tau, theta]


Output()

Sampling 3 chains for 1_000 tune and 10_000 draw iterations (3_000 + 30_000 draws total) took 29 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [mu, tau, theta]


Output()

Sampling 3 chains for 1_000 tune and 10_000 draw iterations (3_000 + 30_000 draws total) took 33 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Bridge sampling estimate of the log marginal likelihood of H0: -37.53794
The percentage errors of the estimation is: 0.31743
Bridge sampling estimate of the log marginal likelihood of H1: -37.79165
The percentage errors of the estimation is: 0.38085
The estimated Bayes factor in favor of H0 over H1 is equal to: 1.28880
